# Refine Spot Detections with (Multi-)Gaussian fit

## User Input

- base directory of dataset
- subpath to CSV file(s) containing candidate spot detections
    - columns containing (pixel) coordinates
    - column containing path of files (must be in same base directory)

## Output

- new subdirectory with CSV files(s) augmented by Gaussian fit parameters

In [ ]:
from pathlib import Path
from tifffile import imread
from tqdm import tqdm
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Ellipse

from calmutils.localization import refine_multi_gaussian_fit
from calmutils.misc.file_utils import get_common_subpath
from calmutils.localization.util import get_ellipse_params


def plot_detections_projection(img, axis, means, sigmas, ax=None, aspect=1, flip=False):

    if ax is None:
        _, ax = plt.subplots()

    means = means[:, [d != axis for d in range(img.ndim)]]
    sigmas = sigmas[:, [d != axis for d in range(img.ndim)]]

    img_proj = img.max(axis)

    if flip:
        img_proj = img_proj.T
        means = means[:,::-1]
        sigmas = sigmas[:, ::-1]

    ax.imshow(img_proj, aspect=aspect)

    for mu, sigma in zip(means, sigmas):
        cov = np.diag(sigma)**2
        a, b, alpha = get_ellipse_params(cov)
        ell = Ellipse(mu[::-1], b, a, angle=alpha, color='red', fill=None, linewidth=3)
        ax.add_artist(ell)

In [ ]:
base_path = "/data/agl_data/NanoFISH/Gabi/GS781_Nanog_RNA-DNA_all/20250708_DNA"

csv_path = "detections_spots0.035_aligned/merge_fixed_filenames_goodonly.csv"

image_path_column = "img"
channel_column = "channel"
coordinate_columns = ["z", "y", "x"]

channels_to_include = (0, 1)

results_subpath = "detections_refined_multigauss"

save_plots = True
visualization_path = "vis"

RESULT_COLUMN_NAMES = [
    "gauss_fit_background",
    "gauss_fit_height",
    "gauss_fit_mu_z",
    "gauss_fit_mu_y",
    "gauss_fit_mu_x",
    "gauss_fit_sigma_z",
    "gauss_fit_sigma_y",
    "gauss_fit_sigma_x"
]

In [ ]:
in_files = sorted(Path(base_path).glob(csv_path))

in_files

In [ ]:
for in_file in in_files:

    print(f"processing {in_file}")

    df = pd.read_csv(in_file)
    result_dfis = []

    for (image_file, channel), dfi in tqdm(df.groupby([image_path_column, channel_column], sort=False)):

        if channel not in channels_to_include:
            continue

        # try to replace differing prefixes
        # of base path specified in this notebook and image paths in table
        # e.g. when running this on mounted data from a different machine than the one used for detection
        _, (prefix_from_base, prefix_table), _ = get_common_subpath(base_path, image_file)
        image_file = image_file.replace(prefix_table, prefix_from_base)

        img = imread(image_file)
        coords = dfi[coordinate_columns].values

        ref_coords, ref_params = refine_multi_gaussian_fit(img, coords, 9)

        # fit did not work -> fill with NaN
        if ref_params is None:
            ref_params = np.full((len(coords), 2 * img.ndim + 2), np.nan)

        dfi[RESULT_COLUMN_NAMES] = ref_params
        result_dfis.append(dfi)

        if save_plots:

            plot_params = ref_params if not np.isnan(ref_params).any() else np.array([]).reshape(-1, 2 * img.ndim + 2)

            plot_out_directory = Path(base_path) / results_subpath / visualization_path
            if not plot_out_directory.exists():
                plot_out_directory.mkdir(parents=True)

            plot_out_file = plot_out_directory / (Path(image_file).stem + "_detections.png")

            fig, axs = plt.subplots(2, 2, figsize=(8,8))
            plot_detections_projection(img, 0, plot_params[:,2:5], plot_params[:,5:], ax=axs[0,0])
            plot_detections_projection(img, 1, plot_params[:,2:5], plot_params[:,5:], ax=axs[1,0], aspect=3)
            plot_detections_projection(img, 2, plot_params[:,2:5], plot_params[:,5:], ax=axs[0,1], aspect=1/3, flip=True)
            fig.delaxes(axs[1,1])
            fig.tight_layout()
            fig.savefig(plot_out_file)
            plt.close()

    result_df = pd.concat(result_dfis)

    out_directory = Path(base_path) / results_subpath
    if not out_directory.exists():
        out_directory.mkdir()

    result_df.to_csv(out_directory / in_file.name)

## Other Stuff

In [ ]:
# table of only good files
if False:
    goods_imgs_df = pd.read_csv(Path(base_path) / "good_imgs.csv")
    good_imgs = goods_imgs_df["img"].str.split("/", expand=True).iloc[:,-1].unique()

    df = pd.read_csv(in_files[0])
    file_stem = df.img.str.rsplit("_", n=2, expand=True)[0].str.split("/", expand=True).iloc[:,-1]

    df_good = df[file_stem.isin(good_imgs)]
    df_good.to_csv(str(in_files[0]).replace(".csv", "_goodonly.csv"), index=None)

In [ ]:
# replace filenames in table (renamed tif directory)
if False:
    df[image_path_column] = df[image_path_column].str.replace("tif_sted_aligned1", "tif_sted_aligned")
    df.to_csv(str(in_file).replace(".csv", "_fixed_filenames.csv"), index=None)